# Galea analysis

This notebook loads a session **with derived signals**, opens the signal browser, and adds analysis GUIs that run on the **browser's time range**.

* **Derived** channels (browser group "Derived"):
  * `EMG<n> env`: EMG linear envelope (muscle activity).
  * `Linear accel |a|`, `Gyro |ω|`, `Pitch`, `Roll`: head motion from IMU fusion.
  * `Heart rate`, `Breathing (PPG)`, `Breathing phase`: computed from the PPG.
* The browser starts clipped to the session's **valid range**, if one was confirmed in `quality_report.ipynb`. Otherwise it shows the whole recording. To analyse a sub-range, zoom to it and press **clip to window**.
* **Pair analysis:** pick channels A and B, a preprocessing and an analysis (cross-correlation or coherence), each with its own options. **run** recomputes on the current range; **export figure** saves a PNG to `exports/`.

In [ ]:
%load_ext autoreload
%autoreload 2
from galea import load_session, resolve_data, valid_range
from galea.browser import SignalBrowser
from galea.pair_gui import PairGUI

DATA = None          # session folder path; None → folder dialog (or $GALEA_DATA if set)

## Load

In [ ]:
s = load_session(resolve_data(DATA), derived=True)
vr = valid_range(s)
print(f"{s.name}: start {s.t0_wall():%Y-%m-%d %H:%M:%S %z}, duration {s.exg.t.iloc[-1]:.1f} s; valid range: "
      + (f"{vr[0]:.1f}–{vr[1]:.1f} s" if vr else "NOT CONFIRMED (whole recording; set it in quality_report.ipynb)"))
for r in s.resp_info:
    print(f"  breathing run {r['t0']:.1f}–{r['t1']:.1f} s: {r['beats']} beats, "
          f"{r['breaths_per_min']:.1f} breaths/min, HR–pulse-amplitude r = {r['hr_amp_r']:.2f}")

## Browser
It starts clipped to the valid range, if confirmed. Press **unclip** to see the whole recording.

In [ ]:
b = SignalBrowser(s, channels=["EEG7", "EOG0", "PPG IR", "Heart rate", "Breathing (PPG)", "Linear accel |a|"])
if vr:
    b.set_clip(*vr)
b

## Pair analysis

In [ ]:
pg = PairGUI(b, "Heart rate", "Breathing (PPG)", analysis="coherence")
pg